# When Should a Small Language Model Say “I Don’t Know”?

### CISC7021 Applied Natural Language Processing — method runs and analysis

**A (Deng Yunfan)** · data pipeline · evaluation protocol · M1 / M3 · two-stage · statistics · figures
**B (Zhang Lu)** · inference interface · M2 · M4 · experiment runs · case analysis

It downloads SQuAD 2.0, builds the fixed 1,000-question val / test splits, loads
Qwen2.5-0.5B-Instruct, and runs every method the report compares:

| Method | Idea | Cost | Owner |
|---|---|---|---|
| **M1** | length-normalised token log-prob of one greedy answer | 1 pass | A |
| **M3** | 5-sample lexical consistency (largest cluster share) | 5 samples | A |
| **Two-stage (ours)** | M1 screens; only borderline questions pay for M3 | 1 + 5p samples | A |
| **M2-conf** | model answers, then self-reports a 0–100 confidence | 1 pass | B |
| **M2-idk** | prompt: “if the answer is not in the context, reply I don’t know” | 1 pass | B |
| **M4** | in-domain LoRA SFT on SQuAD train (750 answerable + 750 refusal) | 1 pass | B |

**Protocol:** thresholds are tuned on **val only**; test is scored once. Coverage is the share
answered; selective accuracy is accuracy on answered; overall accuracy counts correct answers +
correct refusals. The always-abstain baseline is **50.0%** because half the test questions are
unanswerable.

**Default run:** a 5-question end-to-end demo (`RUN_QUICK_DEMO=True`). Every heavier path
(full table, M2, M4 training / inference) is behind its own flag — see the next cell.

## How to run (Colab)

1. `Runtime -> Change runtime type -> T4 GPU`.
2. Clone the repo if needed: `!git clone https://github.com/MC648472/NLP-Abstention-Project.git`
   then `%cd NLP-Abstention-Project`.
3. Run the cells in order. The scorer `src/abstention_eval.py`, the M2/M4 modules in
   `b_methods/`, and `slm_interface.py` are found automatically; a local fallback keeps
   the notebook runnable if `slm_interface.py` is missing.
4. Flags (all default to the cheap path, so “Run all” on a T4 finishes in ~2 minutes):

   | Flag | Where | What it does |
   |---|---|---|
   | `RUN_QUICK_DEMO=True` | §10 | 5-question end-to-end demo (default) |
   | `RUN_FULL_EXPERIMENT=True` | §10 | full 1,000-question table → `results/notebook_main_table.csv` |
   | `RUN_M2=True` | §11 | re-run both M2 variants on the GPU (≈10 min) |
   | `RUN_M4_TRAIN=True` | §12 | retrain the M4 adapter (3.6 min, 9 GB peak) |
   | `RUN_M4_INFER=True` | §12 | re-infer M4 with the committed `models/m4_lora_r50` adapter |
   | all flags `False` | — | cached tables only; CPU is enough, no model download |

In [1]:
# A = Deng Yunfan: data pipeline, evaluation protocol, M1/M3, two-stage, statistics, figures.
# B = Zhang Lu: inference interface, M2, M4, experiment runs, case analysis.

# ============================================================================
# Session setup: on Colab, clone the repo once per session (or pull if it is
# already there). Skipped outside Colab, so a local run is unaffected.
# ============================================================================
import os
import subprocess

_REPO_DIR = "/content/NLP-Abstention-Project"
if os.path.isdir("/content"):
    if not os.path.isdir(_REPO_DIR):
        subprocess.run(["git", "clone", "-b", "final",
                        "https://github.com/MC648472/NLP-Abstention-Project.git",
                        _REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", _REPO_DIR, "pull", "--ff-only"], check=True)
    os.chdir(_REPO_DIR)
    print("repo:", subprocess.run(["git", "-C", _REPO_DIR, "log", "--oneline", "-1"],
                                  capture_output=True, text=True).stdout.strip())

# ============================================================================
# CISC7021 Applied NLP — Abstention in Small Language Models
# A: data / evaluation / M1 / M3 / two-stage; B: M2 / M4 / diagnostics
# Model: Qwen2.5-0.5B-Instruct  |  Data: SQuAD 2.0
# 1,000 questions per split: 500 answerable + 500 unanswerable, seed 42
# Run: Runtime -> T4 GPU -> Run all. Default = 5-question demo.
# Full table: set RUN_FULL_EXPERIMENT=True in the last code cell.
# ============================================================================
import hashlib
import json
import math
import os
import random
import re
import string
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_NAME = "Abstention in Small Language Models"
IN_COLAB = os.path.isdir("/content")

if IN_COLAB:
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive")
    except Exception as exc:
        print(f"Drive mount skipped: {exc}")


def _find_repo_root():
    """Find the repo root from the notebook's cwd, or from /content/* on Colab."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "abstention_eval.py").exists() or (p / "src" / "abstention_eval.py").exists():
            return p
    for p in Path("/content").glob("*/abstention_eval.py"):
        return p.parent
    for p in Path("/content").glob("*/src/abstention_eval.py"):
        return p.parent.parent
    return here


REPO_ROOT = _find_repo_root()
for _p in [REPO_ROOT, REPO_ROOT / "src", REPO_ROOT / "b_methods", Path.cwd()]:
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))


def seed_for_qid(qid):
    """Stable per-question seed: the same qid gets the same samples in any subset."""
    return int(hashlib.md5(str(qid).encode("utf-8")).hexdigest()[:8], 16) % (2**31 - 1)


DATA_DIR = os.environ.get(
    "SQUAD_DIR", "/content/squad_data" if IN_COLAB else str(REPO_ROOT / "squad_data")
)
OUT_DIR = os.environ.get("ABSTENTION_OUT", str(REPO_ROOT / "results"))
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(DATA_DIR, "train-v2.0.json")
DEV_FILE = os.path.join(DATA_DIR, "dev-v2.0.json")

# One directory for the shared items/gold files.  M2/M4 cells below read
# data/items directly, so writing the splits somewhere else used to leave them
# out of sync when the seed changed.
ITEMS_DIR = os.environ.get("ITEMS_DIR", str(REPO_ROOT / "data" / "items"))

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 32
TEMPERATURE = 0.7
N_SAMPLES = 8                 # store 8 samples for M3 / Stage 2
M3_K = 5                      # score the first 5, matching the report
VAL_FRAC = 0.5
MAX_PER_BUCKET = 500
BATCH_SIZE = 8                # batch size for greedy M1
QUICK_N = 5                   # questions in the live demo (keep small on CPU)
RUN_QUICK_DEMO = True         # default: 5-question demo on Colab
RUN_FULL_EXPERIMENT = False   # set True for the 1,000-question table
# (the same two flags are repeated at the bottom of the last code cell; edit there)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | model: {MODEL_NAME} | seed: {SEED}")
print(f"Repo: {REPO_ROOT}")
print(f"Data dir: {os.path.abspath(DATA_DIR)} | output dir: {os.path.abspath(OUT_DIR)}")


repo: 41844f7 Update evaluation code, results and figures; remove legacy F1 artifacts
Mounted at /content/drive
Device: cpu | model: Qwen/Qwen2.5-0.5B-Instruct | seed: 42
Repo: /content/NLP-Abstention-Project
Data dir: /content/squad_data | output dir: /content/NLP-Abstention-Project/results


## 1. Data: SQuAD 2.0

SQuAD 2.0 nests data as `articles -> paragraphs -> qas`. Every QA entry gets flattened into one
record with `qid`, `question`, `context`, `answers` and `is_impossible`. The downloader only runs
when a file is missing, so a pre-uploaded copy in `DATA_DIR` is used as-is.


In [2]:
SQUAD_FILES = {
    TRAIN_FILE: "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json",
    DEV_FILE: "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json",
}


def ensure_squad_files():
    """Download the two SQuAD 2.0 files if they are not on disk yet."""
    import urllib.request

    for path, url in SQUAD_FILES.items():
        if os.path.exists(path) and os.path.getsize(path) > 0:
            continue
        print(f"downloading {url}")
        urllib.request.urlretrieve(url, path)


def load_squad_records(path):
    """Flatten a SQuAD 2.0 json file into a list of records."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    records = []
    for article in raw["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                answers = [a["text"].strip() for a in qa.get("answers", [])
                           if a["text"].strip()]
                records.append({
                    "qid": qa["id"],
                    "context": context,
                    "question": qa["question"].strip(),
                    "answers": answers,
                    "is_impossible": bool(qa.get("is_impossible", False)),
                })
    return records


def dataset_stats(records, name):
    answerable = sum(not r["is_impossible"] for r in records)
    print(f"{name:10s} total={len(records):6d}  answerable={answerable:6d}  "
          f"unanswerable={len(records) - answerable:6d}")


try:
    ensure_squad_files()
except Exception as exc:  # offline and no local copy
    raise RuntimeError(
        f"SQuAD 2.0 files are missing in {os.path.abspath(DATA_DIR)} and could not be "
        "downloaded. Upload train-v2.0.json / dev-v2.0.json there and re-run this cell."
    ) from exc

train_records = load_squad_records(TRAIN_FILE)
dev_records = load_squad_records(DEV_FILE)
dataset_stats(train_records, "train")
dataset_stats(dev_records, "dev")


downloading https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json
downloading https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json
train      total=130319  answerable= 86821  unanswerable= 43498
dev        total= 11873  answerable=  5928  unanswerable=  5945


## 2. Answer normalisation and correctness

Official SQuAD normalisation (lowercase, drop articles / punctuation, collapse whitespace).
A prediction on an answerable question counts as correct when it matches a gold answer by EM;
F1 is kept as a secondary score. On an unanswerable question the correct behaviour is to abstain.

`clean_prediction` trims chatty outputs (first line, "Answer:" prefixes, quotes) before scoring,
which matters for a 0.5B model that sometimes echoes the instruction.


In [3]:
def normalize_answer(text):
    """Lowercase, remove articles, punctuation and extra whitespace."""
    if text is None:
        return ""
    text = str(text).lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = "".join(ch for ch in text if ch not in set(string.punctuation))
    return " ".join(text.split())


def em_score(prediction, gold):
    return int(normalize_answer(prediction) == normalize_answer(gold))


def f1_score(prediction, gold):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()
    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def score_against_golds(prediction, golds):
    em = max((em_score(prediction, g) for g in golds), default=0)
    f1 = max((f1_score(prediction, g) for g in golds), default=0.0)
    return em, f1


def clean_prediction(text):
    """Trim instruction-following noise before grading."""
    text = str(text).strip()
    text = text.split("\n")[0].strip()
    text = re.sub(r"^(the\s+)?answer\s*(is)?\s*[:\-]\s*", "", text, flags=re.IGNORECASE)
    return text.strip(" \"'`.")


# Apostrophes and punctuation are removed by normalize_answer, so every pattern
# below is written in the same normalised space ("don't" -> "dont").
REFUSAL_PATTERNS = [
    "i dont know", "i do not know", "dont know", "do not know",
    "cannot answer", "can not answer", "cant answer",
    "could not answer", "couldnt answer", "unable to answer",
    "not enough information", "insufficient information", "no information",
    "cannot be determined", "cant be determined", "cannot determine",
    "not specified", "not stated", "not mentioned", "unanswerable",
    "no answer", "no relevant information", "i cannot find",
    "do not have enough", "there is no information",
]


def is_abstention(text):
    norm = normalize_answer(text)
    if not norm:
        return True
    return any(pattern in norm for pattern in REFUSAL_PATTERNS)


def forced_correct(record, prediction):
    """Would this answer be correct if the system decided to answer?"""
    if record["is_impossible"]:
        return False
    em, _ = score_against_golds(prediction, record["answers"])
    return em == 1


def judge_record(record, prediction, abstain):
    """Correctness of the final behaviour: answer correctly, or refuse an unanswerable."""
    if record["is_impossible"]:
        return abstain, 0, 0.0
    if abstain:
        return False, 0, 0.0
    em, f1 = score_against_golds(prediction, record["answers"])
    return em == 1, em, f1


## 3. Fixed splits

SQuAD 2.0 ships only `train` and `dev`.

- `train` is used for M4 LoRA fine-tuning.
- `dev` is split by answerability into `val` (all threshold tuning) and `test` (final numbers only).

Each bucket is capped independently at `MAX_PER_BUCKET` questions, and the exact splits are written
to `OUT_DIR` so every method evaluates on the same questions.

The same cell also writes the four item/gold files (`b_items_{val,test}.jsonl` and
`b_gold_{val,test}.jsonl`) to `data/items/` (the same directory the
M2/M4 cells read). Those are exactly the questions to feed to `slm_interface.run_batch`.


In [4]:
def stratified_split(records, val_frac, seed):
    """Split records into val / test while keeping the answerable ratio."""
    rng = random.Random(seed)
    answerable = [r for r in records if not r["is_impossible"]]
    unanswerable = [r for r in records if r["is_impossible"]]
    rng.shuffle(answerable)
    rng.shuffle(unanswerable)

    def cut(items):
        k = int(len(items) * val_frac)
        return items[:k], items[k:]

    a_val, a_test = cut(answerable)
    u_val, u_test = cut(unanswerable)
    return a_val + u_val, a_test + u_test


def cap_buckets(records, max_per_bucket, seed):
    """Cap the answerable and unanswerable buckets independently."""
    rng = random.Random(seed)
    answerable = [r for r in records if not r["is_impossible"]]
    unanswerable = [r for r in records if r["is_impossible"]]
    rng.shuffle(answerable)
    rng.shuffle(unanswerable)
    return answerable[:max_per_bucket] + unanswerable[:max_per_bucket]


val_records, test_records = stratified_split(dev_records, VAL_FRAC, SEED)
val_records = cap_buckets(val_records, MAX_PER_BUCKET, SEED)
test_records = cap_buckets(test_records, MAX_PER_BUCKET, SEED)
dataset_stats(val_records, "val")
dataset_stats(test_records, "test")

for name, split in [("val", val_records), ("test", test_records)]:
    with open(os.path.join(OUT_DIR, f"squad2_{name}.json"), "w", encoding="utf-8") as f:
        json.dump(split, f, ensure_ascii=False, indent=2)
print(f"splits written to {os.path.abspath(OUT_DIR)}")

# The four files B needs for slm_interface.run_batch: exact items + gold.
os.makedirs(ITEMS_DIR, exist_ok=True)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

for name, split in [("val", val_records), ("test", test_records)]:
    items = [{"id": r["qid"], "question": r["question"], "context": r["context"]}
             for r in split]
    gold = [{"id": r["qid"], "question": r["question"], "answers": r["answers"],
             "is_impossible": r["is_impossible"]} for r in split]
    write_jsonl(os.path.join(ITEMS_DIR, f"b_items_{name}.jsonl"), items)
    write_jsonl(os.path.join(ITEMS_DIR, f"b_gold_{name}.jsonl"), gold)
print(f"item/gold files written to {os.path.abspath(ITEMS_DIR)}")


val        total=  1000  answerable=   500  unanswerable=   500
test       total=  1000  answerable=   500  unanswerable=   500
splits written to /content/NLP-Abstention-Project/results
item/gold files written to /content/NLP-Abstention-Project/data/items


## 4. Inference interface

`slm_interface.py` is the shared inference path. The notebook wraps `SLM` in a thin
adapter with a list-based `generate_batch` interface:

- **M1** uses `SLM.generate(question, context)` and takes `avg_logprob` as the confidence;
- **M3** uses `SLM.sample(question, n=8, context=...)` and keeps the first 3/5/8 answers;
- the `SYSTEM_PROMPT` is imported from `slm_interface.py` and never re-defined, so all methods
  (M1/M3 and M2/M4) are evaluated under the same prompt.

If `slm_interface.py` is missing, `LocalGenerator` falls back to the same model and the same prompt
so the notebook still runs standalone; results from the two paths are not mixed in one table.
The sequence confidence for M1 is the length-normalised mean token log-probability of the generated
answer tokens; `prob = exp(confidence)` is its sequence-level probability in `[0, 1]`.


In [5]:
# Load the shared inference interface; fall back to a local model if missing.
try:
    from slm_interface import SLM, SYSTEM_PROMPT
    import slm_interface as _slm_module
    HAS_B_INTERFACE = True
    print(f"slm_interface loaded from {_slm_module.__file__}")
except ImportError:
    SLM = None
    HAS_B_INTERFACE = False
    # Must stay identical to slm_interface.SYSTEM_PROMPT.
    SYSTEM_PROMPT = "Answer the question with a short phrase only. Do not explain."


class BGenerator:
    """Adapter from SLM to the list-based batch interface."""

    def __init__(self):
        self.slm = None

    def load(self):
        if not HAS_B_INTERFACE:
            raise RuntimeError(
                "slm_interface.py was not found. Add it or use "
                "LocalGenerator for a standalone run."
            )
        self.slm = SLM()
        print("using slm_interface.SLM")

    def generate_batch(self, questions, contexts, n_samples=1, return_logprobs=False,
                       do_sample=False, temperature=TEMPERATURE, batch_size=BATCH_SIZE,
                       seeds=None):
        # SLM is per question; keep the list-based batch interface.
        answers, logprobs = [], []
        for index, (question, context) in enumerate(zip(questions, contexts)):
            if n_samples == 1 and not do_sample:
                result = self.slm.generate(question, context)
                answers.append([clean_prediction(result["answer"])])
                if return_logprobs:
                    logprobs.append([float(result["avg_logprob"])])
            else:
                result = self.slm.sample(
                    question, n=n_samples, temperature=temperature,
                    context=context,
                    seed=(seeds[index] if seeds is not None else SEED + index),
                )
                answers.append([clean_prediction(a) for a in result["answers"]])
                if return_logprobs:
                    logprobs.append([0.0] * len(result["answers"]))
        return (answers, logprobs) if return_logprobs else answers

    def generate(self, question, context, n_samples=1, return_logprobs=False,
                 do_sample=False, temperature=TEMPERATURE, seeds=None):
        result = self.generate_batch(
            [question], [context], n_samples=n_samples,
            return_logprobs=return_logprobs, do_sample=do_sample,
            temperature=temperature, seeds=seeds,
        )
        if return_logprobs:
            answer_lists, logprob_lists = result
            return answer_lists[0], logprob_lists[0][0]
        return result[0]


class LocalGenerator:
    """Fallback used only when slm_interface.py is missing.

    It mirrors the prompt and sampling settings; batch generation is used
    for speed.
    """

    def __init__(self, model_name=MODEL_NAME, device=DEVICE):
        self.model_name = model_name
        self.device = device
        self.tokenizer = None
        self.model = None

    def load(self):
        from transformers import AutoModelForCausalLM, AutoTokenizer

        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name, torch_dtype=dtype
        ).to(self.device)
        self.model.eval()
        print(f"loaded {self.model_name} on {self.device} (local fallback)")

    def _prompt(self, question, context):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context: {context}\nQuestion: {question}"},
        ]
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    def _mean_logprob(self, scores, token_ids, row):
        values = []
        for step, token_id in zip(scores, token_ids.tolist()):
            log_probs = torch.log_softmax(step[row], dim=-1)
            values.append(log_probs[token_id].item())
        return sum(values) / len(values) if values else 0.0

    @torch.no_grad()
    def generate_batch(self, questions, contexts, n_samples=1, return_logprobs=False,
                       do_sample=False, temperature=TEMPERATURE, batch_size=BATCH_SIZE,
                       seeds=None):
        all_answers, all_logprobs = [], []
        sample = do_sample or n_samples > 1
        if sample and seeds is not None:
            batch_size = 1  # one question per batch, so each qid keeps its own seed

        for start in range(0, len(questions), batch_size):
            qs = questions[start:start + batch_size]
            cs = contexts[start:start + batch_size]
            prompts = [self._prompt(q, c) for q, c in zip(qs, cs)]
            enc = self.tokenizer(prompts, return_tensors="pt", padding=True)
            enc = {k: v.to(self.device) for k, v in enc.items()}
            prompt_len = enc["input_ids"].shape[1]

            kwargs = dict(
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=sample,
                num_return_sequences=n_samples,
                pad_token_id=self.tokenizer.eos_token_id,
                output_scores=True,
                return_dict_in_generate=True,
            )
            if sample:
                kwargs.update(temperature=temperature, top_p=0.95, top_k=50)

            if sample and seeds is not None:
                torch.manual_seed(int(seeds[start]))
            out = self.model.generate(**enc, **kwargs)
            for b in range(len(qs)):
                answers, logprobs = [], []
                for r in range(n_samples):
                    row = b * n_samples + r
                    gen_ids = out.sequences[row, prompt_len:]
                    eos = (gen_ids == self.tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
                    if len(eos):
                        gen_ids = gen_ids[:eos[0]]
                    text = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
                    answers.append(clean_prediction(text))
                    if return_logprobs:
                        logprobs.append(self._mean_logprob(out.scores, gen_ids, row))
                all_answers.append(answers)
                if return_logprobs:
                    all_logprobs.append(logprobs)

        return (all_answers, all_logprobs) if return_logprobs else all_answers

    @torch.no_grad()
    def generate(self, question, context, n_samples=1, return_logprobs=False,
                 do_sample=False, temperature=TEMPERATURE, seeds=None):
        result = self.generate_batch(
            [question], [context], n_samples=n_samples,
            return_logprobs=return_logprobs, do_sample=do_sample,
            temperature=temperature, batch_size=1, seeds=seeds,
        )
        if return_logprobs:
            answer_lists, logprob_lists = result
            return answer_lists[0], logprob_lists[0][0]
        return result[0]


def make_generator():
    """Use slm_interface when available, otherwise the local fallback."""
    return BGenerator() if HAS_B_INTERFACE else LocalGenerator()


gen = make_generator()


slm_interface loaded from /content/NLP-Abstention-Project/slm_interface.py


## 5. Metric suite and calibration

Every prediction record carries `qid`, gold fields, `answer`, `abstain`, `correct`, `confidence`
(higher = more confident, used for AUROC / AUPRC and thresholding), `prob` (confidence mapped to
`[0, 1]`, used for ECE / Brier) and `n_samples` / `latency_s` for cost.

For M1, `prob = exp(mean token log-prob)`; for M3, `prob = consistency`. Calibration metrics are
therefore computed on a real probability scale instead of clipping negative log-probabilities to 0.


In [6]:
# Metrics come from the shared abstention_eval module.
import os
import sys

# Locate abstention_eval.py in the repo or handoff directory.
_REPO = globals().get("REPO_ROOT", os.getcwd())
_CANDIDATE_DIRS = [os.path.join(str(_REPO), "src"), str(_REPO),
                   globals().get("ITEMS_DIR"),
                   os.path.join(os.getcwd(), "src"), os.getcwd(),
                   "/content/drive/MyDrive/Colab Notebooks", "/content/drive/MyDrive"]
for _d in _CANDIDATE_DIRS:
    if _d and os.path.isdir(_d) and os.path.isfile(os.path.join(_d, "abstention_eval.py")):
        if _d not in sys.path:
            sys.path.insert(0, _d)
        break
else:
    raise ImportError("abstention_eval.py not found.")

from abstention_eval import (  # noqa: E402
    coverage, selective_accuracy, risk, correct_refusal_rate, over_refusal_rate,
    auroc_auprc, forced_correct, judge, evaluate, format_metrics,
    tune_at_coverage, apply_threshold, tune_by_accuracy, tune_two_stage_at_coverage,
    brier, ece_score,
)
expected_calibration_error = ece_score
brier_score = brier


def _as_eval_record(p):
    # Notebook records use `qid`; the shared scorer keys on `id` + `forced_correct`.
    q = dict(p)
    q["id"] = q.get("id") or q.get("qid")
    if "forced_correct" not in q:
        q["forced_correct"] = forced_correct(q, q.get("answer", ""))
    return q


def evaluate_predictions(preds, include_calibration=True):
    metrics = evaluate([_as_eval_record(p) for p in preds])
    if not include_calibration:
        metrics.pop("ece", None)
        metrics.pop("brier", None)
    return metrics


## 6. M1 — single-pass token log-probability confidence

One greedy pass per question. Confidence is the length-normalised mean log-probability of the
generated answer tokens; `prob = exp(confidence)`. Abstain when the score is below `tau_m1`
(tuned on val) **or** when the model explicitly refuses (a verbal "I don't know" with high
confidence should not be turned into an answer).

The full pipeline uses `tune_at_coverage` fixed-coverage thresholds on val; the notebook's
`tune_threshold` wrapper only falls back to overall accuracy when no target coverage is given.


In [7]:
def _gold_should_abstain(pred):
    if pred["is_impossible"]:
        return True
    return not forced_correct(pred, pred["answer"])


def tune_threshold(preds_val, target_coverage=None):
    """Select a threshold on val.

    With target_coverage, use fixed coverage; otherwise maximize overall
    behaviour accuracy. Returns (tau, score), where score is the validation
    behaviour accuracy or the achieved coverage.
    """
    recs = [_as_eval_record(p) for p in preds_val]
    if target_coverage is not None:
        return tune_at_coverage(recs, target_coverage)
    return tune_by_accuracy(recs)




def apply_m1_threshold(preds, tau):
    out = []
    for p in preds:
        p = dict(p)
        p["abstain"] = p.get("verbal_abstain", False) or p["confidence"] < tau
        p["correct"], p["em"], p["f1"] = judge_record(p, p["answer"], p["abstain"])
        out.append(p)
    return out


def run_m1(generator, records, batch_size=BATCH_SIZE):
    """One greedy pass per question; confidence = mean token log-probability."""
    questions = [r["question"] for r in records]
    contexts = [r["context"] for r in records]
    seeds = [seed_for_qid(r["qid"]) for r in records]
    start = time.time()
    answers, logprobs = generator.generate_batch(
        questions, contexts, n_samples=1, return_logprobs=True,
        do_sample=False, batch_size=batch_size, seeds=seeds,
    )
    per_question = (time.time() - start) / max(1, len(records))

    preds = []
    for i, record in enumerate(records):
        answer = answers[i][0]
        confidence = logprobs[i][0]
        verbal = is_abstention(answer)
        correct, em, f1 = judge_record(record, answer, verbal)
        preds.append({
            "qid": record["qid"],
            "question": record["question"],
            "context": record["context"],
            "answers": record["answers"],
            "is_impossible": record["is_impossible"],
            "answer": answer,
            "abstain": verbal,
            "verbal_abstain": verbal,
            "confidence": float(confidence),
            "prob": float(math.exp(confidence)),
            "correct": correct,
            "em": em,
            "f1": f1,
            "n_samples": 1,
            "latency_s": per_question,
        })
    return preds


## 7. M3 — multi-sample consistency

Sample `N_SAMPLES` answers at temperature 0.7, cluster them by normalised lexical match /
containment, and take the largest-cluster share as the confidence score. The representative answer
is the largest cluster's member.

A question whose majority answer is an explicit refusal is treated as an abstention regardless of
the consistency value; otherwise M3 would answer "I don't know" five times with consistency 1.0.
In the full pipeline, `tau_c` is tuned on val for the same fixed coverage target as M1.


In [8]:
def cluster_answers(answers):
    """Group answers by normalised exact match / containment."""
    clusters = []  # list of (normalised representative, [raw answers])
    for raw in answers:
        norm = normalize_answer(raw)
        placed = False
        for representative, members in clusters:
            if norm == representative or (
                norm and representative and (norm in representative or representative in norm)
            ):
                members.append(raw)
                placed = True
                break
        if not placed:
            clusters.append((norm, [raw]))
    return [members for _, members in clusters]


def consistency_score(answers):
    if not answers:
        return 0.0, 0
    clusters = cluster_answers(answers)
    return max(len(c) for c in clusters) / len(answers), len(clusters)


def majority_answer(answers):
    clusters = cluster_answers(answers)
    return max(clusters, key=len)[0]


def run_m3(generator, records, n_samples=N_SAMPLES, batch_size=max(1, BATCH_SIZE // 2),
           seed_mode="qid"):
    questions = [r["question"] for r in records]
    contexts = [r["context"] for r in records]
    if seed_mode == "index":
        # Match the cached run: seed = 0 + index.
        seeds = list(range(len(records)))
    else:
        seeds = [seed_for_qid(r["qid"]) for r in records]
    start = time.time()
    sampled = generator.generate_batch(
        questions, contexts, n_samples=n_samples,
        return_logprobs=False, do_sample=True, batch_size=batch_size,
        seeds=seeds,
    )
    per_question = (time.time() - start) / max(1, len(records))

    preds = []
    for record, answers in zip(records, sampled):
        consistency, n_clusters = consistency_score(answers)
        answer = majority_answer(answers)
        refusal = is_abstention(answer)
        correct, em, f1 = judge_record(record, answer, refusal)
        preds.append({
            "qid": record["qid"],
            "question": record["question"],
            "context": record["context"],
            "answers": record["answers"],
            "is_impossible": record["is_impossible"],
            "answer": answer,
            "sample_answers": answers,
            "n_clusters": n_clusters,
            "abstain": refusal,
            "verbal_abstain": refusal,
            "confidence": float(consistency),
            "prob": float(consistency),
            "correct": correct,
            "em": em,
            "f1": f1,
            "n_samples": n_samples,
            "latency_s": per_question,
        })
    return preds


def rescore_m3(preds, k):
    """Recompute M3 from the first k stored samples (no new generation)."""
    out = []
    for p in preds:
        answers = p["sample_answers"][:k]
        consistency, n_clusters = consistency_score(answers)
        answer = majority_answer(answers)
        refusal = is_abstention(answer)
        q = dict(p)
        q.update({
            "answer": answer,
            "abstain": refusal,
            "verbal_abstain": refusal,
            "confidence": float(consistency),
            "prob": float(consistency),
            "n_samples": k,
            "n_clusters": n_clusters,
        })
        q["correct"], q["em"], q["f1"] = judge_record(p, answer, refusal)
        out.append(q)
    return out


def apply_m3_threshold(preds, tau_c):
    out = []
    for p in preds:
        p = dict(p)
        p["abstain"] = p.get("verbal_abstain", False) or p["confidence"] < tau_c
        p["correct"], p["em"], p["f1"] = judge_record(p, p["answer"], p["abstain"])
        out.append(p)
    return out


## 8. Two-stage compute-aware abstainer

1. **Stage 1 (M1 screening).** One greedy pass gives `s1`.
   - `s1 >= tau_high`: answer directly.
   - `s1 <= tau_low`: abstain directly.
   - otherwise **borderline**.
2. **Stage 2 (M3 refinement).** Only borderline questions are sampled `N_SAMPLES` times; the
   consistency threshold `tau_c` decides answer vs abstain.

Cost is recorded per question: 1 sequence for high/low, `1 + N_SAMPLES` for borderline.

For tuning, `tune_two_stage` reuses **cached** M1 and M3 val predictions and simulates every
pair of decile thresholds, so the search costs one M1 pass and one M3 pass rather than a full
pipeline per candidate. The grid and objective are the same ones used by the report:
`tau_low` / `tau_high` are deciles of the val M1 score, `tau_c ∈ {0.2, 0.4, 0.6, 0.8, 1.0}`,
val coverage must be within 0.02 of the target, and ties go to fewer samples.


In [9]:
def simulate_two_stage(m1_preds, m3_preds, tau_low, tau_high, tau_c):
    """Build two-stage predictions from cached M1 / M3 val outputs (no generation)."""
    m3_by_qid = {p["qid"]: p for p in m3_preds}
    preds = []
    for p1 in m1_preds:
        s1 = p1["confidence"]
        stage1_answer = p1["answer"]
        if s1 >= tau_high:
            stage = "high"
            answer = stage1_answer
            abstain = p1.get("verbal_abstain", False)
            used = 1
            consistency = None
            confidence = p1["prob"]
        elif s1 <= tau_low:
            stage = "low"
            answer = stage1_answer
            abstain = True
            used = 1
            consistency = None
            confidence = p1["prob"]
        else:
            stage = "borderline"
            p3 = m3_by_qid[p1["qid"]]
            answer = p3["answer"]
            consistency = p3["confidence"]
            abstain = p3.get("verbal_abstain", False) or consistency < tau_c
            used = 1 + p3["n_samples"]
            confidence = consistency

        correct, em, f1 = judge_record(p1, answer, abstain)
        preds.append({
            "qid": p1["qid"],
            "question": p1["question"],
            "context": p1["context"],
            "answers": p1["answers"],
            "is_impossible": p1["is_impossible"],
            "answer": answer,
            "stage1_score": s1,
            "stage": stage,
            "consistency": consistency,
            "abstain": abstain,
            "confidence": float(confidence),
            "prob": float(confidence),
            "correct": correct,
            "em": em,
            "f1": f1,
            "n_samples": used,
            "latency_s": (p1.get("latency_s") or 0.0)
            + ((m3_by_qid[p1["qid"]].get("latency_s") or 0.0)
               if stage == "borderline" else 0.0),
        })
    return preds


def tune_two_stage(m1_val, m3_val, target_coverage=0.5, quantiles=None):
    """Val-only: pick (tau_low, tau_high, tau_c) for a target coverage.

    Delegates to the shared scorer (the same code that writes the report table),
    so the notebook and run_fixed_coverage.py cannot drift. `tau_low` / `tau_high`
    are deciles of the val M1 score (the pre-registered grid in
    docs/two_stage_spec.md); `tau_c` is in {0.2, 0.4, 0.6, 0.8, 1.0}. Objective =
    overall behaviour accuracy among configs whose val coverage is within 2 points
    of the target; ties go to fewer samples. The old
    `selective_accuracy - 0.5 * over_refusal_rate` objective is gone: it is not
    comparable across methods (selective accuracy already depends on coverage)
    and it ignores correctly refused questions.
    """
    gold_val = {r["qid"]: {"is_impossible": r["is_impossible"], "answers": r["answers"]}
                for r in val_records}
    cfg, acc, cov = tune_two_stage_at_coverage(
        [_as_eval_record(p) for p in m1_val],
        [_as_eval_record(p) for p in m3_val],
        gold_val, target_coverage, quantiles=quantiles,
    )
    return {"config": cfg, "metrics": None, "objective": float(acc), "coverage": float(cov)}


def run_two_stage(generator, records, tau_low, tau_high, tau_c,
                  n_samples=N_SAMPLES, batch_size=BATCH_SIZE):
    """Run the real two-stage pipeline on a record list."""
    start = time.time()
    m1_preds = run_m1(generator, records, batch_size=batch_size)
    borderline = [p for p in m1_preds if tau_low < p["confidence"] < tau_high]
    m3_preds = run_m3(generator, borderline, n_samples=n_samples) if borderline else []

    preds = []
    m3_by_qid = {p["qid"]: p for p in m3_preds}
    for p1 in m1_preds:
        s1 = p1["confidence"]
        if s1 >= tau_high:
            stage, answer = "high", p1["answer"]
            abstain = p1.get("verbal_abstain", False)
            used, consistency, confidence = 1, None, p1["prob"]
        elif s1 <= tau_low:
            stage, answer = "low", p1["answer"]
            abstain = True
            used, consistency, confidence = 1, None, p1["prob"]
        else:
            p3 = m3_by_qid[p1["qid"]]
            stage, answer = "borderline", p3["answer"]
            consistency = p3["confidence"]
            abstain = p3.get("verbal_abstain", False) or consistency < tau_c
            used, confidence = 1 + p3["n_samples"], consistency

        correct, em, f1 = judge_record(p1, answer, abstain)
        preds.append({
            "qid": p1["qid"],
            "question": p1["question"],
            "context": p1["context"],
            "answers": p1["answers"],
            "is_impossible": p1["is_impossible"],
            "answer": answer,
            "stage1_score": s1,
            "stage": stage,
            "consistency": consistency,
            "abstain": abstain,
            "confidence": float(confidence),
            "prob": float(confidence),
            "correct": correct,
            "em": em,
            "f1": f1,
            "n_samples": used,
            "latency_s": None,
        })

    per_question = (time.time() - start) / max(1, len(records))
    for p in preds:
        p["latency_s"] = per_question
    return preds


## 9. Curves and ablations

- `coverage_accuracy_curve` is a diagnostic sweep; the report figure uses
  `abstention_eval.risk_coverage_curve` through `src/make_figures.py`.
- `sample_count_ablation` runs the M3 k=3/5/8 ablation; the report's Stage-2 k ablation
  is `src/ablate_two_stage.py`.


In [10]:
def coverage_accuracy_curve(preds, n_points=25):
    ordered = sorted(preds, key=lambda p: p["confidence"])
    curve = []
    for k in range(1, len(ordered) + 1):
        tau = ordered[k - 1]["confidence"]
        decided = []
        for p in ordered:
            q = dict(p)
            q["abstain"] = q.get("verbal_abstain", False) or q["confidence"] < tau
            q["correct"], _, _ = judge_record(q, q["answer"], q["abstain"])
            decided.append(q)
        metrics = evaluate_predictions(decided)
        curve.append({"threshold": float(tau), "coverage": metrics["coverage"],
                      "selective_accuracy": metrics["selective_accuracy"],
                      "risk": metrics["risk"]})
    step = max(1, len(curve) // n_points)
    return curve[::step]


def sample_count_ablation(generator, val_records, test_records, counts=(3, 5, 8)):
    """Sample the largest k once per split, then slice to 3/5/8."""
    max_k = max(counts)
    val_full = run_m3(generator, val_records, n_samples=max_k)
    test_full = run_m3(generator, test_records, n_samples=max_k)
    rows = []
    for k in counts:
        val_preds = rescore_m3(val_full, k)
        tau_c, _ = tune_threshold(val_preds)
        test_preds = apply_m3_threshold(rescore_m3(test_full, k), tau_c)
        metrics = evaluate_predictions(test_preds)
        metrics["n_samples_setting"] = k
        metrics["tau_c"] = tau_c
        rows.append(metrics)
    return rows


## 10. Running the experiments

Two modes, both in the last code cell:

- **Quick demo (default)**: 5 val questions end-to-end. M1 screens, borderline
  questions go to 5-sample M3, and the result is printed as a small table. This
  is the mode for the live Colab demonstration (about 1-2 minutes on T4).
- **Full experiment**: 1,000 questions per split, 8 samples stored and the first
  5 scored, fixed coverage 0.3 and 0.5, thresholds tuned on val only. This writes
  `results/notebook_main_table.csv` in the same schema as the report; the curated
report table stays at `results/main_table_fixed_coverage.csv`.

The report numbers come from the **full** mode; the demo is only for showing the
pipeline. If you just want to display the already-computed table, set both flags
to `False` and run `show_cached_results()`.


In [11]:
import pandas as pd
from IPython.display import display


def preds_to_dataframe(preds):
    rows = []
    for p in preds:
        rows.append({
            "qid": p["qid"],
            "is_impossible": p["is_impossible"],
            "answer": p["answer"],
            "abstain": p["abstain"],
            "confidence": p["confidence"],
            "prob": p.get("prob"),
            "correct": p["correct"],
            "em": p.get("em"),
            "f1": p.get("f1"),
            "stage": p.get("stage"),
            "n_samples": p["n_samples"],
            "latency_s": p.get("latency_s"),
        })
    return pd.DataFrame(rows)


def split_error_cases(preds):
    buckets = {"correct_answer": [], "incorrect_answer": [],
               "unnecessary_refusal": [], "missed_refusal": []}
    for p in preds:
        if p["is_impossible"]:
            if p["abstain"]:
                continue
            buckets["missed_refusal"].append(p)
        elif p["abstain"]:
            buckets["unnecessary_refusal"].append(p)
        elif p["correct"]:
            buckets["correct_answer"].append(p)
        else:
            buckets["incorrect_answer"].append(p)
    return buckets


def save_run(method_name, preds, metrics, out_dir=OUT_DIR):
    df = preds_to_dataframe(preds)
    df.to_csv(os.path.join(out_dir, f"{method_name}_preds.csv"), index=False)
    with open(os.path.join(out_dir, f"{method_name}_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)
    return df


def _metric_row(method, target, tau, val_cov, metrics, note=""):
    return {
        "method": method,
        "target_cov": target,
        "val_tau": round(float(tau), 4) if tau is not None else "",
        "val_cov": round(float(val_cov), 4),
        "test_cov": round(float(metrics["coverage"]), 4),
        "selective_acc": round(float(metrics["selective_accuracy"]), 4),
        "overall_acc": round(float(metrics["behaviour_accuracy"]), 4),
        "correct_refusal": round(float(metrics["correct_refusal_rate"]), 4),
        "over_refusal": round(float(metrics["over_refusal_rate"]), 4),
        "avg_samples": round(float(metrics["avg_samples"]), 2),
        "avg_latency_s": round(float(metrics.get("avg_latency_s") or 0.0), 3),
        "note": note,
    }


def run_full_pipeline(generator, val_records, test_records, targets=(0.3, 0.5),
                      run_ablations=False):
    """Fixed-coverage evaluation: store 8 samples, score 5, tune on val only."""
    generator.load()

    print("M1: val ...")
    m1_val = run_m1(generator, val_records)
    print("M1: test ...")
    m1_test = run_m1(generator, test_records)

    print(f"M3: val ({N_SAMPLES} samples) ...")
    m3_val_full = run_m3(generator, val_records, n_samples=N_SAMPLES,
                         seed_mode="index")
    print(f"M3: test ({N_SAMPLES} samples) ...")
    m3_test_full = run_m3(generator, test_records, n_samples=N_SAMPLES,
                          seed_mode="index")
    m3_val = rescore_m3(m3_val_full, M3_K)
    m3_test = rescore_m3(m3_test_full, M3_K)

    rows, summary = [], {}
    for target in targets:
        # ---- M1 -----------------------------------------------------------
        tau_m1, val_cov1 = tune_threshold(m1_val, target)
        m1_val_t = apply_m1_threshold(m1_val, tau_m1)
        m1_test_t = apply_m1_threshold(m1_test, tau_m1)
        m1m = evaluate_predictions(m1_test_t)
        rows.append(_metric_row("M1", target, tau_m1, val_cov1, m1m))
        save_run(f"m1_test_cov{target}", m1_test_t, m1m)

        # ---- M3 -----------------------------------------------------------
        tau_c, val_cov3 = tune_threshold(m3_val, target)
        m3_val_t = apply_m3_threshold(m3_val, tau_c)
        m3_test_t = apply_m3_threshold(m3_test, tau_c)
        m3m = evaluate_predictions(m3_test_t)
        unreachable = abs(val_cov3 - target) > 0.05
        note = (f"M3 cannot reach {target}: min val coverage {val_cov3:.3f}"
                if unreachable else "")
        rows.append(_metric_row("M3 (k=5)", target, tau_c, val_cov3, m3m, note))
        save_run(f"m3_k5_test_cov{target}", m3_test_t, m3m)

        # ---- two-stage (simulated from the full M1 / M3 predictions) -------
        best = tune_two_stage(m1_val, m3_val, target)
        cfg = best["config"]
        ts_val = simulate_two_stage(m1_val, m3_val, *cfg)
        ts_test = simulate_two_stage(m1_test, m3_test, *cfg)
        tsm = evaluate_predictions(ts_test)
        note = f"tau_low={cfg[0]:.3f}, tau_high={cfg[1]:.3f}, tau_c={cfg[2]:.1f}"
        rows.append(_metric_row("Two-stage", target, cfg[0], best["coverage"], tsm, note))
        save_run(f"twostage_test_cov{target}", ts_test, tsm)

        summary[str(target)] = {"m1": m1m, "m3": m3m, "two_stage": tsm,
                                "two_stage_cfg": list(cfg)}

    table = pd.DataFrame(rows)
    # Write the notebook's table; the curated table stays in results/.
    table.to_csv(os.path.join(OUT_DIR, "notebook_main_table.csv"), index=False)
    with open(os.path.join(OUT_DIR, "notebook_results_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    if run_ablations:
        print("ablation: sample counts {3, 5, 8} ...")
        ablation = pd.DataFrame(sample_count_ablation(generator, val_records, test_records))
        ablation.to_csv(os.path.join(OUT_DIR, "m3_sample_count_ablation.csv"), index=False)

    print("\n=== test table ===")
    display(table)
    return table


def run_quick_demo(generator, records, n=QUICK_N):
    """5-question end-to-end demo for the presentation; thresholds are the
    full-val 0.3 operating point, so this is illustrative, not a new result."""
    generator.load()
    # records are answerable-first; keep the last two (unanswerable) so the
    # demo shows both an answer and a correct refusal.
    subset = records[:n - 2] + records[-2:] if n >= 3 else records[:n]
    m1 = run_m1(generator, subset)
    m3 = rescore_m3(run_m3(generator, subset, n_samples=5), 5)
    ts = simulate_two_stage(m1, m3, -0.131, -0.085, 0.8)

    rows = []
    for p in ts:
        rows.append({
            "question": p["question"][:70],
            "m1_conf": round(p["stage1_score"], 3),
            "stage": p["stage"],
            "answer": p["answer"],
            "decision": "ABSTAIN" if p["abstain"] else "ANSWER",
            "correct": p["correct"],
        })
    df = pd.DataFrame(rows)
    display(df)
    print("demo metrics (5 questions, not the report table):")
    print(evaluate_predictions(ts))
    return df


def show_cached_results():
    """Show the report tables without loading the model (fast path for the talk)."""
    for name in ["main_table_fixed_coverage.csv", "m2_m4_table.csv",
                 "ci_table.csv", "pairwise_ci.csv"]:
        path = os.path.join(OUT_DIR, name)
        if os.path.exists(path):
            print("\n== " + name + " ==")
            display(pd.read_csv(path))


RUN_FULL_EXPERIMENT = False
RUN_QUICK_DEMO = True

if RUN_FULL_EXPERIMENT:
    main_table = run_full_pipeline(gen, val_records, test_records)
elif RUN_QUICK_DEMO:
    demo_df = run_quick_demo(gen, val_records, QUICK_N)
else:
    show_cached_results()


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

using slm_interface.SLM


,question,m1_conf,stage,answer,decision,correct
0,How many nations are within the Amazon Basin?,-0.230,low,nine,ABSTAIN,False
1,What welding process was demonstrated in 1901?,-0.085,high,oxyacetylene welding,ANSWER,True
2,How is packet switching charecterized,-1.125,low,Packet-switched,ABSTAIN,False
3,What's the Latin name for Germany?,-0.996,low,Austronesus,ABSTAIN,True
4,What kind of auricles do cilia have?,-0.686,low,four,ABSTAIN,True


demo metrics (5 questions, not the report table):
{'n': 5, 'coverage': 0.2, 'selective_accuracy': 1.0, 'risk': 0.0, 'correct_refusal_rate': 1.0, 'over_refusal_rate': 0.6666666666666666, 'correct_answer_refusal_rate': 0.5, 'wrong_answer_refusal_rate': 1.0, 'auroc': 1.0, 'auprc': 1.0, 'ece': 0.29684827474814285, 'brier': 0.10887629175331684, 'behaviour_accuracy': 0.6, 'avg_samples': 1.0, 'avg_latency_s': 3.5555588722229006}


## 11. M2 — verbal confidence and prompted refusal (B)

Two zero-shot variants, same 1,000-question test split:

- **M2-conf**: the model answers and then prints `Answer: … / Confidence: 0-100` (4 few-shot
  turns show the format). The confidence is thresholded on **val** exactly like M1.
- **M2-idk**: the prompt says “if the answer is not in the context, reply exactly: I don’t know”.
  No threshold: the model decides by itself.

Both are scored with `src/abstention_eval.py`, and both keep `avg_logprob` on every record so the
behaviour / cost columns stay comparable with M1.

`RUN_M2=True` regenerates the generations (GPU); `False` shows the cached rows from
`results/m2_m4_table.csv`.

In [12]:
# ---------------------------------------------------------------------------
# 11. M2 - verbal confidence ("conf") and prompted refusal ("idk")
# ---------------------------------------------------------------------------
import subprocess
import sys

for _p in ["b_methods", "src", "."]:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

import m2_verbal as m2

B_ITEMS = {s: m2.load_jsonl(f"data/items/b_items_{s}.jsonl") for s in ("val", "test")}
B_GOLD = {s: f"data/items/b_gold_{s}.jsonl" for s in ("val", "test")}
RUN_M2 = False          # True -> re-run both variants on the GPU (~10 min)

if RUN_M2:
    from slm_interface import SLM
    m2_slm = SLM()
    for version, name in [("conf", "m2conf"), ("idk", "m2idk")]:
        for split in ("val", "test"):
            m2.run_m2(m2_slm, B_ITEMS[split],
                      f"data/outputs_b/slm_{name}_{split}.jsonl", version, overwrite=True)
    # Fixed-coverage scoring.
    subprocess.run([sys.executable, "src/score_m2_m4.py"], check=True)
    m2.diagnostics("data/outputs_b/slm_m2conf_val.jsonl", B_GOLD["val"])

_t = pd.read_csv("results/m2_m4_table.csv")
display(_t[_t.method.str.startswith("M2")])

print("M2-conf: only 11 distinct confidence values on val (0.2, 0.7-0.99) -> AUROC 0.602, "
      "48.0% overall; the 0.3 target is unreachable (lowest 0.389).")
print("M2-idk: refuses 228/1000 times; 31.4% correct refusal at 77.2% coverage.")

,method,target_cov,tau,val_cov,test_cov,selective_acc,behaviour_acc,correct_refusal,over_refusal,auroc,avg_samples,note
0,M2-conf,0.3,0.95,0.379,0.389,0.4087,0.480,0.642,0.580,0.6023,1.0,NaN
1,M2-conf,0.5,0.90,0.465,0.467,0.4176,0.480,0.570,0.496,0.6023,1.0,NaN
2,M2-idk,natural,NaN,0.765,0.772,0.2098,0.319,0.314,0.142,NaN,1.0,refuses by itself (no threshold); AUROC not re...


M2-conf: only 11 distinct confidence values on val (0.2, 0.7-0.99) -> AUROC 0.602, 48.0% overall; the 0.3 target is unreachable (lowest 0.389).
M2-idk: refuses 228/1000 times; 31.4% correct refusal at 77.2% coverage.


## 12. M4 — in-domain LoRA SFT (B)

M4 is **supervised fine-tuning**, not threshold tuning: 1,500 examples from the SQuAD 2.0
**train** split, 750 answerable (target = the gold answer) + 750 unanswerable (target =
`I don’t know`), 2 epochs of LoRA on `q/k/v/o_proj` (r=8, α=16, lr 2e-4, batch 4, max_len 512).
Training takes 3.6 min and peaks at 9.0 GB on a T4.

Two consequences the report states explicitly:

- the gain includes **in-domain QA fine-tuning**, not only abstention ability — on the
  answerable questions it *does* answer, M4 is right 80.0% of the time, against 58.2% forced
  accuracy for M1;
- it still **over-refuses 42.0%** of answerable questions.

The training pool is disjoint from val / test (`|pool ∩ val| = |pool ∩ test| = 0`, sha256 in
`results/m4_train_pool_check.json`). Because `avg_logprob` here is confidence in *its own output*
(including “I don’t know”), M4 does **not** enter the cross-method AUROC / AUPRC / ECE / Brier
comparison — only behaviour and cost.

In [13]:
# ---------------------------------------------------------------------------
# 12. M4 - in-domain LoRA SFT: pool -> training set -> (train) -> score
# ---------------------------------------------------------------------------
import subprocess

import m4_lora as m4

RUN_M4_TRAIN = False    # True -> retrain the adapter (3.6 min, 9.0 GB peak on a T4)
RUN_M4_INFER = False    # True -> re-infer with models/m4_lora_r50

print(subprocess.run([sys.executable, "b_methods/check_train_overlap.py"],
                     capture_output=True, text=True,
                     encoding="utf-8", errors="replace").stdout.strip())

pool = m2.load_jsonl("data/m4_train_pool.jsonl")
examples = m4.build_train_examples(pool, refusal_ratio=0.5, max_n=1500, seed=0)
print("first answerable example  ->", examples[0]["target"])
print("first refusal example     ->",
      next(e["target"] for e in examples if not e["answerable"]))

if RUN_M4_TRAIN:
    m4.train_lora(examples, "models/m4_lora_r50", r=8, alpha=16, epochs=2, lr=2e-4, bs=4)
if RUN_M4_INFER:
    m4.infer_m4("models/m4_lora_r50", B_ITEMS["test"], "data/outputs_b/slm_m4_test.jsonl", overwrite=True)

m4_metrics, m4_scored = m4.score_m4("data/outputs_b/slm_m4_test.jsonl", B_GOLD["test"],
                                    "results/m4_test_scored.jsonl",
                                    base_pred_path="data/outputs_b/slm_m1_test.jsonl")
m4_cases = m4.case_list(m4_scored, B_GOLD["test"], kind="over_refusal",
                        base_pred_path="data/outputs_b/slm_m1_test.jsonl", k=5)
m4_missed = m4.case_list(m4_scored, B_GOLD["test"], kind="missed_refusal", k=5)
print("M4 metrics above; cases in results/m4_cases.json.")

pool: 5000 records (2500 answerable + 2500 unanswerable), 5000 unique ids
val : 1000 ids   test: 1000 ids
|pool ∩ val|  = 0
|pool ∩ test| = 0
(deep check skipped: squad_data/train-v2.0.json not found; the two intersection counts above are computed from the committed files)

PASS: wrote results/m4_train_pool_check.json
训练样本: 可答 750 + 拒答 750 = 1500 (拒答占比 50%)
first answerable example  -> October 1982
first refusal example     -> I don't know
n=1000 | coverage=0.370 | selective_accuracy=0.627 | risk=0.373 | correct_refusal_rate=0.840 | over_refusal_rate=0.420 | correct_answer_refusal_rate=0.340 | wrong_answer_refusal_rate=0.531 | auroc=0.338 | auprc=0.167 | ece=0.694 | brier=0.679 | behaviour_accuracy=0.652 | avg_samples=1.000 | avg_latency_s=0.162
over_refusal: 共 99 题, 返回前 5 题
missed_refusal: 共 80 题, 返回前 5 题
M4 metrics above; cases in results/m4_cases.json.


## 13. Refusal-share ablation: r30 / r50 / r70 (cached)

The share of refusal examples in the SFT set is the one knob that trades coverage against
over-refusal. All three generations are committed, so this table is CPU-only.

In [14]:
# ---------------------------------------------------------------------------
# 13. Refusal-share ablation - scored from the committed generations
# ---------------------------------------------------------------------------
subprocess.run([sys.executable, "src/ablate_m4_ratio.py"], check=True)
display(pd.read_csv("results/m4_ratio_ablation.csv"))

,ratio,adapter,coverage,selective_point,selective_boot_mean,selective_ci_low,selective_ci_high,overall_point,overall_boot_mean,overall_ci_low,overall_ci_high,correct_refusal,over_refusal
0,0.3,r30,0.516,0.5388,0.5389,0.4942,0.5810,0.634,0.6340,0.603,0.664,0.712,0.256
1,0.5,r50,0.370,0.6270,0.6270,0.5753,0.6762,0.652,0.6520,0.621,0.682,0.840,0.420
2,0.7,r70,0.264,0.6591,0.6595,0.6016,0.7165,0.629,0.6291,0.599,0.658,0.910,0.562


## 14. Diagnostics: what the confidence signals can and cannot tell us

Two **threshold-free** diagnostics; both read the cached generations and run on CPU:

- **AUROC labelled by “would the forced answer be correct”**, plus the mean `avg_logprob` of the
  three question buckets (answerable & correct / answerable & wrong / unanswerable). The closer the
  “wrong” and “unanswerable” means are, the less the signal can separate *not knowing the answer*
  from *knowing it but answering wrong*.
- **Conditional refusal rate**: on answerable questions that M1 gets right versus wrong, how often
  does a given method refuse? This decomposes where over-refusal comes from.

M4 appears here as well, but its `avg_logprob` is confidence in *its own output* (including the
string “I don’t know”), so an AUROC below 0.5 is an artefact of what the score measures — not
evidence that M4 is worse. In the report this stays a within-method diagnostic and never enters the
cross-method table.

In [15]:
# ---------------------------------------------------------------------------
# 14. Threshold-free diagnostics (CPU, cached generations)
# ---------------------------------------------------------------------------
import numpy as np

import abstention_eval as ev

_DGOLD = {s: ev.load_jsonl(f"data/items/b_gold_{s}.jsonl") for s in ("val", "test")}
_DGID = {s: ev.index_by_id(_DGOLD[s]) for s in _DGOLD}


def _refused(rec):
    """Scored records carry an `abstain` flag; raw generations carry the refusal text."""
    return bool(rec["abstain"]) if "abstain" in rec else ev.is_abstention(rec["answer"])


def threshold_free(pred_path, label, split="test"):
    """AUROC labelled by 'would the forced answer be correct', plus per-bucket log-prob means."""
    pred = ev.index_by_id(ev.load_jsonl(pred_path))
    gold, gid = _DGOLD[split], _DGID[split]
    answerable = [g for g in gold if not g["is_impossible"]]
    forced = np.mean([ev.forced_correct(g, pred[g["id"]]["answer"]) for g in answerable])
    answered = [g for g in answerable if not _refused(pred[g["id"]])]
    acc_answered = (np.mean([ev.forced_correct(g, pred[g["id"]]["answer"]) for g in answered])
                    if answered else float("nan"))
    preds = ev.m1_preds(gold, pred)
    for p in preds:
        p["correct"] = ev.forced_correct(gid[p["id"]], p["answer"])
    auroc, auprc = ev.auroc_auprc(preds)
    buckets = {"answerable & correct": [], "answerable & wrong": [], "unanswerable": []}
    for g in gold:
        a = pred[g["id"]]["answer"]
        key = ("unanswerable" if g["is_impossible"]
               else "answerable & correct" if ev.forced_correct(g, a)
               else "answerable & wrong")
        buckets[key].append(pred[g["id"]]["avg_logprob"])
    print(f"[{label}] forced acc on all answerable: {forced:.1%} | "
          f"acc on the answerable it answers: {acc_answered:.1%} ({len(answered)} answered) | "
          f"AUROC {auroc:.3f} | AUPRC {auprc:.3f}")
    for key, v in buckets.items():
        print(f"   {key:22s} n={len(v):4d}  mean avg_logprob = {np.mean(v):+.3f}")


def conditional_refusal(pred_path, label):
    """Of the answerable questions M1 gets right / wrong, how many does this method refuse?"""
    pred = ev.index_by_id(ev.load_jsonl(pred_path))
    base = ev.index_by_id(ev.load_jsonl("data/outputs_b/slm_m1_test.jsonl"))
    right, wrong = [0, 0], [0, 0]
    for i, g in _DGID["test"].items():
        if g["is_impossible"] or i not in base or i not in pred:
            continue
        target = right if ev.forced_correct(g, base[i]["answer"]) else wrong
        target[0] += int(_refused(pred[i]))
        target[1] += 1
    show = lambda a: f"{a[0]}/{a[1]} = {a[0] / a[1]:.1%}" if a[1] else "n/a"
    print(f"[{label}] refused although M1 was right: {show(right)} | "
          f"refused when M1 was wrong: {show(wrong)}")


threshold_free("data/outputs_b/slm_m1_test.jsonl", "M1")
threshold_free("data/outputs_b/slm_m4_test.jsonl", "M4")
print()
for _label, _path in [("M1 @0.5 cov", "results/m1_test_cov0.5_scored.jsonl"),
                      ("M4", "data/outputs_b/slm_m4_test.jsonl"),
                      ("M2-idk", "data/outputs_b/slm_m2idk_test.jsonl")]:
    conditional_refusal(_path, _label)
print()
print("\nM1's avg_logprob separates 'answerable & correct' (-0.19) from 'answerable & wrong' "
      "(-0.47), but 'unanswerable' (-0.41) sits almost on top of 'wrong' — a threshold can catch "
      "'unsure', it cannot catch 'sure and there is no answer'.")
print("M4's three means are inverted (unanswerable -0.06 looks the most 'confident') because its "
      "avg_logprob scores its own output, including \"I don't know\"; that is why its AUROC of 0.338 "
      "stays out of the cross-method comparison.")


[M1] forced acc on all answerable: 58.2% | acc on the answerable it answers: 58.2% (500 answered) | AUROC 0.739 | AUPRC 0.575
   answerable & correct   n= 291  mean avg_logprob = -0.192
   answerable & wrong     n= 209  mean avg_logprob = -0.469
   unanswerable           n= 500  mean avg_logprob = -0.406
[M4] forced acc on all answerable: 46.4% | acc on the answerable it answers: 80.0% (290 answered) | AUROC 0.338 | AUPRC 0.167
   answerable & correct   n= 232  mean avg_logprob = -0.126
   answerable & wrong     n= 268  mean avg_logprob = -0.109
   unanswerable           n= 500  mean avg_logprob = -0.057

[M1 @0.5 cov] refused although M1 was right: 77/291 = 26.5% | refused when M1 was wrong: 141/209 = 67.5%
[M4] refused although M1 was right: 99/291 = 34.0% | refused when M1 was wrong: 111/209 = 53.1%
[M2-idk] refused although M1 was right: 39/291 = 13.4% | refused when M1 was wrong: 32/209 = 15.3%


M1's avg_logprob separates 'answerable & correct' (-0.19) from 'answerable & wrong' (

## 15. Live demo

One question end to end: confidence -> route -> decision.
Thresholds are the val-tuned 0.3-coverage operating point
(`results/fixed_coverage_config.json`): `tau_low=-0.131`, `tau_high=-0.085`, `tau_c=0.8`.

The three cases cover: answered correctly / correctly refused / answered although it should refuse.


In [16]:
# ============================================================================
# 12. Live demo - one question end to end (for the presentation video)
# Thresholds: val-tuned operating point for 0.3 coverage
# ============================================================================
TAU_LOW, TAU_HIGH, TAU_C = -0.131, -0.085, 0.8

DEMO_CASES = [
    {
        "label": "answerable",
        "question": "What is the name of the desert city?",
        "context": ("Many locals and tourists frequent the southern California coast for its "
                    "popular beaches, and the desert city of Palm Springs is popular for its "
                    "resort feel and nearby open spaces."),
        "gold": "Palm Springs",
        "is_impossible": False,
    },
    {
        "label": "unanswerable, should refuse",
        "question": "What is the ULCA mascot?",
        "context": ("College sports are also popular in southern California. The UCLA Bruins "
                    "and the USC Trojans both field teams in NCAA Division I in the Pac-12 "
                    "Conference, and there is a longtime rivalry between the schools."),
        "gold": None,
        "is_impossible": True,
    },
    {
        "label": "unanswerable, model answers anyway",
        "question": "Who was pioneer and studied specific complexity measure in 1948?",
        "context": ("Earlier papers studying problems solvable by Turing machines with specific "
                    "bounded resources include John Myhill's definition of linear bounded automata "
                    "(Myhill 1960), Raymond Smullyan's study of rudimentary sets (1961), as well as "
                    "Hisao Yamada's paper on real-time computations (1962). Somewhat earlier, Boris "
                    "Trakhtenbrot (1956), a pioneer in the field from the USSR, studied another "
                    "specific complexity measure."),
        "gold": None,
        "is_impossible": True,
    },
]


def demo(case):
    """Run one question through the two-stage abstainer and print the decision."""
    q, ctx = case["question"], case["context"]
    answers, s1 = gen.generate(q, ctx, n_samples=1, return_logprobs=True, do_sample=False)
    a1 = answers[0]

    if s1 >= TAU_HIGH:
        route, decision, final = "Stage 1: s1 >= tau_high -> answer", "ANSWER", a1
    elif s1 <= TAU_LOW:
        route, decision, final = "Stage 1: s1 <= tau_low -> refuse", "ABSTAIN", "I don't know"
    else:
        extra = gen.generate(q, ctx, n_samples=5, return_logprobs=False, do_sample=True)
        cons, _ = consistency_score(extra)
        route = "Stage 2: borderline -> 5 samples, consistency=%.1f" % cons
        if cons >= TAU_C:
            decision, final = "ANSWER", majority_answer(extra)
        else:
            decision, final = "ABSTAIN", "I don't know"

    if decision == "ABSTAIN":
        verdict = "correct refusal" if case["is_impossible"] else "over-refusal"
    elif case["is_impossible"]:
        verdict = "wrong answer (should have refused)"
    else:
        ok = normalize_answer(final) == normalize_answer(case["gold"] or "")
        verdict = "correct" if ok else "incorrect"

    print("=" * 76)
    print("[%s]" % case["label"])
    print("Q: " + q)
    print("Context: " + ctx[:100] + "...")
    print("-" * 76)
    print("confidence (mean token log-prob): %+.3f" % s1)
    print("route    : " + route)
    print("model    : " + final)
    print("decision : " + decision)
    print("verdict  : " + verdict)
    print("=" * 76)
    print()


if getattr(gen, "slm", None) is None and getattr(gen, "model", None) is None:
    gen.load()

for case in DEMO_CASES:
    demo(case)


[answerable]
Q: What is the name of the desert city?
Context: Many locals and tourists frequent the southern California coast for its popular beaches, and the des...
----------------------------------------------------------------------------
confidence (mean token log-prob): -0.016
route    : Stage 1: s1 >= tau_high -> answer
model    : Palm Springs
decision : ANSWER
verdict  : correct

[unanswerable, should refuse]
Q: What is the ULCA mascot?
Context: College sports are also popular in southern California. The UCLA Bruins and the USC Trojans both fie...
----------------------------------------------------------------------------
confidence (mean token log-prob): -1.480
route    : Stage 1: s1 <= tau_low -> refuse
model    : I don't know
decision : ABSTAIN
verdict  : correct refusal

[unanswerable, model answers anyway]
Q: Who was pioneer and studied specific complexity measure in 1948?
Context: Earlier papers studying problems solvable by Turing machines with specific bounded resource